In [12]:
import pathlib

import numpy as np
import pandas as pd

In [13]:
data_dir = pathlib.Path() / "../../data/raw/biolog/phenotypes"

In [14]:
pheno_ch = pd.read_csv(data_dir / "p_CH_phenotypes.tsv", index_col=0, sep="\t").dropna(axis=1, how="all")
pheno_pmi = pd.read_csv(data_dir / "p_PMI_phenotypes.tsv", index_col=0, sep="\t").dropna(axis=1, how="all")
pheno_leaf = pd.read_csv(data_dir / "p_LEAF_phenotypes.tsv", index_col=0, sep="\t").dropna(axis=1, how="all")

In [15]:
names_ch = sorted(list(set(pheno_ch.columns)))
names_pmi = sorted(list(set(pheno_pmi.columns)))
names_leaf = sorted(list(set(pheno_leaf.columns)))

In [18]:
# TODO: Standardize names
prefixes_remove = ["Carbon-{a|b|d|g|m|p}-", "{a|b|d|g|m|p}-", "Carbon-"]
characters_remove = ["-{a|b|d|g|m|p}-", " "]
suffixes_remove = ["Acid"]
prefixes_add = ["D-", "L-"]
# Convert to lower case

In [17]:
n_names = max(len(names_ch), len(names_pmi), len(names_leaf))
names_data = {
    "CH": names_ch + ["" for _ in range(n_names-len(names_ch))],
    "PMI": names_pmi + ["" for _ in range(n_names-len(names_pmi))],
    "Leaf": names_leaf + ["" for _ in range(n_names-len(names_leaf))],
}
names_df = pd.DataFrame(names_data)
names_df.to_csv(data_dir / "phenotype_names.tsv", sep="\t", index=False)
names_df

,CH,PMI,Leaf
0,Carbon-Acetic-Acid,"1,2-Propanediol",Acetate
1,Carbon-Acetoacetic-Acid,"2,3-Butanediol",Alanine
2,Carbon-Citric-Acid,"2,3-Butanone",Arabinose
3,Carbon-D-Arabitol,2-Aminoethanol,Arginine
4,Carbon-D-Aspartic-Acid,2-Deoxy Adenosine,Asparagine
...,...,...,...
186,,i-Erythritol,
187,,m-Hydroxy Phenyl Acetic Acid,
188,,m-Inositol,
189,,m-Tartaric Acid,


In [43]:
def clean_name(raw_name):
    name = raw_name.lower()
    prefixes_remove = ["carbon-"]
    suffixes_remove = ["-acid", "acid"]
    for prefix in prefixes_remove:
        name = name.replace(prefix, "")
    # for character in characters_remove:
    #     name = name.replace(character, "")
    for suffix in suffixes_remove:
        name = name.replace(suffix, "")
    return name.lower()

In [44]:
names_df.apply(lambda x: x.apply(clean_name))

,CH,PMI,Leaf
0,acetic,"1,2-propanediol",acetate
1,acetoacetic,"2,3-butanediol",alanine
2,citric,"2,3-butanone",arabinose
3,d-arabitol,2-aminoethanol,arginine
4,d-aspartic,2-deoxy adenosine,asparagine
...,...,...,...
186,,i-erythritol,
187,,m-hydroxy phenyl acetic,
188,,m-inositol,
189,,m-tartaric,


In [51]:
ch_names = [clean_name(c) for c in names_df.CH if c]
leaf_names = [clean_name(c) for c in names_df.Leaf if c]
pmi_names = [clean_name(c) for c in names_df.PMI if c]

In [52]:
from fuzzywuzzy import fuzz

In [53]:
# Direct comparison
set(ch_names) & set(leaf_names) & set(pmi_names)

{'glycerol', 'maltose', 'sucrose'}

In [54]:
# Fuzzy comparison
for leaf_name, leaf_fullname in zip(leaf_names, names_df.Leaf):
    for ch_name, ch_fullname in zip(ch_names, names_df.CH):
        if fuzz.WRatio(leaf_name, ch_name) > 90:
            for pmi_name, pmi_fullname in zip(pmi_names, names_df.PMI):
                if fuzz.WRatio(leaf_name, pmi_name) > 90:
                    print(f"leaf={leaf_fullname}, ch={ch_fullname}, pmi={pmi_fullname}")

leaf=Alanine, ch=Carbon-L-Alanine, pmi=D-Alanine
leaf=Alanine, ch=Carbon-L-Alanine, pmi=L-Alanine
leaf=Arginine, ch=Carbon-L-Arginine, pmi=L-Arginine
leaf=Cellobiose, ch=Carbon-D-Cellobiose, pmi=D-Cellobiose
leaf=Fructose, ch=Carbon-D-Fructose, pmi=D-Fructose
leaf=Galactose, ch=Carbon-D-Galactose, pmi=D-Galactose
leaf=Glucose, ch=Carbon-D-Glucose, pmi=L-Glucose
leaf=Glycerol, ch=Carbon-Glycerol, pmi=Glycerol
leaf=Histidine, ch=Carbon-L-Histidine, pmi=L-Histidine
leaf=Maltose, ch=Carbon-Maltose, pmi=Maltose
leaf=Mannitol, ch=Carbon-D-Mannitol, pmi=D-Mannitol
leaf=Mannose, ch=Carbon-D-Mannose, pmi=D-Mannose
leaf=Serine, ch=Carbon-D-Serine, pmi=D-Serine
leaf=Serine, ch=Carbon-D-Serine, pmi=L-Serine
leaf=Serine, ch=Carbon-L-Serine, pmi=D-Serine
leaf=Serine, ch=Carbon-L-Serine, pmi=L-Serine
leaf=Sucrose, ch=Carbon-Sucrose, pmi=Sucrose
leaf=Trehalose, ch=Carbon-D-Trehalose, pmi=D-Trehalose


In [56]:
names_df.PMI[names_df.PMI.str.endswith("Glucose")]

9      3-Methyl Glucose
96            L-Glucose
165         a-D-Glucose
Name: PMI, dtype: object